In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import copy

print("PyTorch version:", torch.__version__)
print("All libraries imported successfully!")

PyTorch version: 2.11.0+cpu
All libraries imported successfully!


In [2]:
DATASET_PATH = 'C:/Users/Lenovo/Desktop/color'
IMG_SIZE     = 224
BATCH_SIZE   = 32
EPOCHS       = 10
LR           = 0.001
NUM_CLASSES  = 38
SEED         = 42

print("Config set successfully!")

Config set successfully!


In [3]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

print("Augmentation pipeline created!")

Augmentation pipeline created!


In [4]:
full_dataset = datasets.ImageFolder(DATASET_PATH)

total      = len(full_dataset)
train_size = int(0.70 * total)
val_size   = int(0.15 * total)
test_size  = total - train_size - val_size

train_data, val_data, test_data = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_data.dataset.transform = train_transforms
val_data.dataset.transform   = val_transforms
test_data.dataset.transform  = val_transforms

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Total  : {total}")
print(f"Train  : {train_size}")
print(f"Val    : {val_size}")
print(f"Test   : {test_size}")

Total  : 54305
Train  : 38013
Val    : 8145
Test   : 8147


In [5]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes=38):
        super(CustomCNN, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = CustomCNN(num_classes=NUM_CLASSES).to(device)
print(model)
print(f"\nUsing device: {device}")

CustomCNN(
  (conv1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=100352, out_features=512, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=512, out_features=38, bias=True)
  )
)

Using device: cpu


In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print("Loss      : CrossEntropyLoss")
print("Optimizer : Adam")
print("LR        :", LR)

Loss      : CrossEntropyLoss
Optimizer : Adam
LR        : 0.001


In [ ]:
class EarlyStopping:
    def __init__(self, patience=3):
        self.patience   = patience
        self.counter    = 0
        self.best_loss  = None
        self.best_model = None
        self.stop       = False

    def __call__(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss:
            self.best_loss  = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

early_stopping = EarlyStopping(patience=3)
train_losses, val_losses         = [], []
train_accuracies, val_accuracies = [], []

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs    = model(images)
            loss       = criterion(outputs, labels)
            val_loss  += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc = train_correct / train_size
    val_acc   = val_correct   / val_size
    t_loss    = train_loss    / len(train_loader)
    v_loss    = val_loss      / len(val_loader)

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {t_loss:.4f} Train Acc: {train_acc:.4f} Val Loss: {v_loss:.4f} Val Acc: {val_acc:.4f}")

    early_stopping(v_loss, model)
    if early_stopping.stop:
        print("Early stopping triggered!")
        break

model.load_state_dict(early_stopping.best_model)
print("\nTraining complete!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_accuracies, label='Train Accuracy', color='blue')
axes[0].plot(val_accuracies,   label='Val Accuracy',   color='orange')
axes[0].set_title('Training vs Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(train_losses, label='Train Loss', color='blue')
axes[1].plot(val_losses,   label='Val Loss',   color='orange')
axes[1].set_title('Training vs Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print("Saved: training_curves.png")

In [ ]:
model.eval()
test_correct = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs       = model(images)
        test_correct += (outputs.argmax(1) == labels).sum().item()

test_accuracy = test_correct / test_size
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

torch.save(model.state_dict(), 'custom_cnn_plantvillage.pth')
print("Model saved!")